In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nilani2002/disaster-dataset/notes.json
/kaggle/input/datasets/nilani2002/disaster-dataset/classes.txt
/kaggle/input/datasets/nilani2002/disaster-dataset/labels/652434f2-Image_0585_20260110_121441.txt
/kaggle/input/datasets/nilani2002/disaster-dataset/labels/c82be4b8-Image_0736_20260110_123716.txt
/kaggle/input/datasets/nilani2002/disaster-dataset/labels/4f025bb4-Image_1233_20260112_121638.txt
/kaggle/input/datasets/nilani2002/disaster-dataset/labels/3fc49cef-Image_0084_20260110_110121.txt
/kaggle/input/datasets/nilani2002/disaster-dataset/labels/cb4c7fba-Image_1175_20260112_115903.txt
/kaggle/input/datasets/nilani2002/disaster-dataset/labels/85f0b2e1-Image_0765_20260110_124223.txt
/kaggle/input/datasets/nilani2002/disaster-dataset/labels/ef820297-Image_0652_20260110_122257.txt
/kaggle/input/datasets/nilani2002/disaster-dataset/labels/7344f34f-Image_0322_20260110_113553.txt
/kaggle/input/datasets/nilani2002/disaster-dataset/labels/695b4919-Image_0393_20260110_1141

In [1]:
!pip install ultralytics -q
import os, glob, zipfile, random, shutil, yaml
from ultralytics import RTDETR
from IPython.display import FileLink

print("📦 Checking dataset readiness...")

# 1. Dataset Checking / Auto-Setup
final_dataset_path = '/kaggle/working/dataset'
yaml_path = '/kaggle/working/dataset/data.yaml'

if not os.path.exists(yaml_path):
    print("🔄 Dataset path empty, re-creating data structure...")
    input_dir = '/kaggle/input'
    extract_path = '/kaggle/working/dataset_raw'
    zip_files = glob.glob(f"{input_dir}/**/*.zip", recursive=True)

    if zip_files:
        os.makedirs(extract_path, exist_ok=True)
        with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        search_base = extract_path
    else:
        search_base = input_dir

    images_dir = glob.glob(f"{search_base}/**/images", recursive=True)[0]
    labels_dir = glob.glob(f"{search_base}/**/labels", recursive=True)[0]

    for folder in ['train/images', 'train/labels', 'val/images', 'val/labels']:
        os.makedirs(os.path.join(final_dataset_path, folder), exist_ok=True)

    all_images = [f for f in os.listdir(images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    random.seed(42)
    random.shuffle(all_images)
    split_index = int(len(all_images) * 0.8)

    def move_files(files, split_name):
        for filename in files:
            shutil.copy(os.path.join(images_dir, filename), os.path.join(final_dataset_path, split_name, 'images', filename))
            label_filename = os.path.splitext(filename)[0] + '.txt'
            if os.path.exists(os.path.join(labels_dir, label_filename)):
                shutil.copy(os.path.join(labels_dir, label_filename), os.path.join(final_dataset_path, split_name, 'labels', label_filename))

    move_files(all_images[:split_index], 'train')
    move_files(all_images[split_index:], 'val')

    yaml_content = {
        'path': '/kaggle/working/dataset',
        'train': 'train/images',
        'val': 'val/images',
        'nc': 4,
        'names': ['debris', 'landslide', 'structures', 'uprooted_tree']
    }
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_content, f, default_flow_style=False)

print("✅ Dataset configuration confirmed!")
print("🚀 Starting RT-DETR Vision Transformer Training (30 Epochs)...")

# 2. RT-DETR Model Load and Training
rtdetr_model = RTDETR("rtdetr-l.pt")

results_rtdetr = rtdetr_model.train(
    data="/kaggle/working/dataset/data.yaml", 
    epochs=30,                          
    imgsz=640,                          
    batch=16,           # Optimal memory allocation for GPU T4
    device=0,          
    workers=4,
    name="rtdetr_results"
)

print("🎉 RT-DETR Training Completed Successfully!")

# 3. Create Zip file for download
shutil.make_archive('/kaggle/working/rtdetr_disaster_results', 'zip', '/kaggle/working/runs/detect/rtdetr_results')

print("✅ Zip created successfully! Download from below link:")
FileLink(r'rtdetr_disaster_results.zip')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
📦 Checking dataset readiness...
🔄 Dataset path empty, re-creating data structure...
✅ Dataset configuration confirmed!
🚀 Starting RT-DETR Vision Transformer Training (30 Epochs)...
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_pas

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30      12.3G      0.745      5.992     0.7838         28        640: 100% ━━━━━━━━━━━━ 68/68 2.3s/it 2:392.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.5s/it 13.2s.8ss
                   all        270        415      0.628      0.245      0.122     0.0616

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/30      12.5G     0.5263     0.9874     0.5378         27        640: 100% ━━━━━━━━━━━━ 68/68 2.0s/it 2:161.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.4it/s 6.5s0.8ss
                   all        270        415      0.162       0.13       0.11      0.045

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/30      12.5G     0.2688     0.3913     0.3195         13        640: 100% ━━━━━━━━━━━━ 68/68 1.9s/it 2:061.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.4it/s 6.4s0.8ss
                   all        270        415      0.886      0.859      0.907      0.605

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30      12.5G     0.2615     0.3707     0.3163         12        640: 100% ━━━━━━━━━━━━ 68/68 1.9s/it 2:081.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.4it/s 6.4s0.8ss
                   all        270        415      0.877      0.846      0.902      0.609

30 epochs completed in 1.143 hours.
Optimizer stripped from /kaggle/working/runs/detect/rtdetr_results/weights/last.pt, 66.2MB
Optimizer stripped from /kaggle/working/runs/detect/rtdetr_results/weights/best.pt, 66.2MB

Validating /kaggle/working/runs/detect/rtdetr_results/weights/best.pt...
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rt-detr-l summary: 310 layers, 31,991,960 parameters, 0 gradients, 103.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 2.0s/it 17.9s1.2s
                   all        270        415       0.88      0.846      0

/kaggle/working/rtdetr_disaster_results.zip